> **How to use this notebook.** The text is the cheat sheet. The code cells run the same equations in plain NumPy so you can change a number and see what happens. Run top to bottom: later cells reuse names from earlier ones (`silu`, `rms_norm`, the config values). The first code cell reads your model's `config.json` (set `QWEN_CONFIG` to point elsewhere) and falls back to the published values if the file is missing. Only NumPy is needed. The animation cells use inline HTML and JavaScript through `IPython.display`, so they work in Jupyter and VS Code with no extra packages; run each one to see it move.


# Qwen3.8-27B on the ANE: architecture cheat sheet

A working reference for reading and extending this model in Forge. Every formula is taken from the reference
implementation (`transformers/models/qwen3_5/modeling_qwen3_5.py`) or from Forge's graph builder
(`coreai/qwen38_coreai_build.py`). Line numbers are for the checkout this was written against; re-check them after
pulling. Things marked **(unverified)** are my reading, not something I ran.


In [1]:
# Animations below are small HTML + JavaScript widgets shown with IPython.display.
# They need Jupyter (or VS Code's notebook viewer) and no extra packages. Run a cell to see its animation.
import uuid
try:
    from IPython.display import HTML, display
    def show_html(html):
        display(HTML(html.replace("__ID__", "a" + uuid.uuid4().hex[:8])))
except ImportError:
    def show_html(html):
        print("IPython is not installed here: open this notebook in Jupyter or VS Code to see the animation.")

## 0. Map of the code

| What | Where |
| --- | --- |
| Reference math (PyTorch, BF16, easy to read) | `.venv/lib/python3.11/site-packages/transformers/models/qwen3_5/modeling_qwen3_5.py` |
| Reference DeltaNet recurrence | `torch_recurrent_gated_delta_rule` (line 438), layer class `Qwen3_5GatedDeltaNet` (504) |
| Reference attention | `Qwen3_5Attention` (749), RoPE (666), MLP (824), RMSNorm (841), decoder layer (861) |
| Forge ANE graph: DeltaNet | `coreai/qwen38_coreai_build.py` `GDNW` (406), `verify` (464), `prefill` (484) |
| Forge ANE graph: attention | `AttnW` (527) |
| Forge ANE graph: one layer, one entry point | `LayerW` (781), `Entry` (819), `Head` (902) |
| Quantized linear layer | `QConv` (218), rotations `Hadamard` (264) |
| Runtime that drives the chunks | `scripts/qwen38_coreai_model.py`, KV helpers `scripts/qwen38_kv_cache.py` |
| Server, drafting and acceptance | `scripts/qwen38_server.py` (`draft_cycle`), `docs/SPECULATIVE_DECODING.md` |
| Quantization (GPTQ, QAT, lookup tables) | `docs/QUANTIZATION.md`, `scripts/qwen38_gptq_27b.py` |


## 1. The model in numbers

From `config.json` (text part):

| Quantity | Value | Symbol used below |
| --- | --- | --- |
| Layers | 64 (pattern: 3 DeltaNet, 1 attention, repeated 16 times) | |
| Hidden size | 5,120 | `H` |
| MLP inner size | 17,408 | `F` |
| Vocabulary | 248,320 | `V` |
| Attention: query heads / KV heads / head size | 24 / 4 / 256 | `nh`, `nkv`, `hd` |
| Attention: query-head groups per KV head | 24 / 4 = 6 | `grp` |
| Attention: rotary dims | `0.25 * 256 = 64` | `rot` |
| DeltaNet: key heads / value heads | 16 / 48 | `nk`, `nv` |
| DeltaNet: key head size / value head size | 128 / 128 | `dk`, `dv` |
| DeltaNet: total key width / value width | 2,048 / 6,144 | `kd`, `vd` |
| DeltaNet: conv width over q,k,v channels | `2*2048 + 6144 = 10,240`, kernel 4 | `cdim` |
| RMSNorm epsilon | 1e-6 | `eps` |
| RoPE | theta 10,000,000, interleaved mRoPE sections [11, 11, 10] | |
| Native max positions | 262,144 | |
| Embeddings tied to the output head? | no (`tie_word_embeddings: false`) | |
| Activation | SiLU | |

The config also lists `mtp_num_hidden_layers: 1`. Forge speeds up decoding with its own DFlash2 drafter instead;
I did not see Forge use the MTP weights **(unverified)**.


In [2]:
import json, os
from pathlib import Path
import numpy as np

CFG_PATH = Path(os.environ.get("QWEN_CONFIG",
                Path.home() / "Models/anemll-forge-qwen3.8-27B-192K/model/config.json"))
try:
    cfg = json.loads(CFG_PATH.read_text())
    cfg = cfg.get("text_config", cfg)
    print("config read from", CFG_PATH)
except FileNotFoundError:
    # the published values, used when the model files are not on this machine
    cfg = dict(hidden_size=5120, intermediate_size=17408, vocab_size=248320, num_hidden_layers=64,
               num_attention_heads=24, num_key_value_heads=4, head_dim=256, linear_num_key_heads=16,
               linear_num_value_heads=48, linear_key_head_dim=128, linear_value_head_dim=128,
               linear_conv_kernel_dim=4, rope_parameters={"partial_rotary_factor": 0.25},
               layer_types=(["linear_attention"] * 3 + ["full_attention"]) * 16)
    print("model files not found, using the published values")

H, F, V = cfg["hidden_size"], cfg["intermediate_size"], cfg["vocab_size"]
nh, nkv, hd = cfg["num_attention_heads"], cfg["num_key_value_heads"], cfg["head_dim"]
nk, nv, dk, dv = (cfg[k] for k in ("linear_num_key_heads", "linear_num_value_heads",
                                   "linear_key_head_dim", "linear_value_head_dim"))
kd, vd = nk * dk, nv * dv
cdim = 2 * kd + vd
rot = int(hd * cfg["rope_parameters"]["partial_rotary_factor"])
layer_types = cfg["layer_types"]
n_att, n_gdn = layer_types.count("full_attention"), layer_types.count("linear_attention")

print(f"hidden {H}, MLP inner {F}, vocabulary {V}")
print(f"layers: {n_gdn} DeltaNet + {n_att} attention; attention layers at {[i for i, t in enumerate(layer_types) if t == 'full_attention'][:4]} ...")
print(f"attention: {nh} query heads, {nkv} KV heads (groups of {nh // nkv}), head size {hd}, rotary dims {rot}")
print(f"DeltaNet: key width {kd}, value width {vd}, conv channels {cdim}")

# numbers quoted in the cheat sheet, recomputed
mlp_per_layer = 3 * H * F
print(f"\nMLP weights per layer {mlp_per_layer:,}  ->  all layers {mlp_per_layer * len(layer_types) / 1e9:.2f} billion")
print(f"DeltaNet state per layer {nv * dk * dv:,}  ->  all DeltaNet layers {n_gdn * nv * dk * dv / 1e6:.1f} million (fixed)")
kv_numbers = n_att * nkv * hd * 2
print(f"attention cache per token {kv_numbers:,} numbers = {(kv_numbers + n_att * nkv * 2 * 2 / 1) / 1024:.2f} KiB at 8 bits + FP16 scales")
assert kv_numbers == 32768 and mlp_per_layer == 267_386_880

config read from /Users/sa4drao/Models/anemll-forge-qwen3.8-27B-192K/model/config.json
hidden 5120, MLP inner 17408, vocabulary 248320
layers: 48 DeltaNet + 16 attention; attention layers at [3, 7, 11, 15] ...
attention: 24 query heads, 4 KV heads (groups of 6), head size 256, rotary dims 64
DeltaNet: key width 2048, value width 6144, conv channels 10240

MLP weights per layer 267,386,880  ->  all layers 17.11 billion
DeltaNet state per layer 786,432  ->  all DeltaNet layers 37.7 million (fixed)
attention cache per token 32,768 numbers = 32.25 KiB at 8 bits + FP16 scales


## 2. Notation

- `x` is the residual stream: one vector of `H = 5120` numbers per token.
- `T` is the number of tokens processed in one call (8 for verification, 64 or 32 for prefill).
- `S` is a DeltaNet state matrix. `K, V` are attention caches.
- `*` is matrix multiply, `⊙` is element-wise multiply, `σ` is sigmoid, `silu(u) = u * σ(u)`.


## 3. The whole forward pass

```
h = Embed[token_ids]                                   # (T, 5120), table is 248,320 x 5,120
for layer i in 0..63:
    r = h
    h = RMSNorm_1(h)                                   # input_layernorm
    h = DeltaNet(h)  if layer_types[i] == linear_attention  (3 of every 4)
        Attention(h) if layer_types[i] == full_attention    (every 4th: layers 3, 7, 11, ...)
    h = r + h                                          # residual 1
    r = h
    h = RMSNorm_2(h)                                   # post_attention_layernorm
    h = MLP(h)
    h = r + h                                          # residual 2
h = RMSNorm_final(h)
logits = h[last row] * W_head                          # (248,320,)  one score per word
```

Source: `Qwen3_5DecoderLayer.forward` (reference line 874), `LayerW` and `Entry._forward` in Forge (819 onward).
The token mixer (DeltaNet or attention) is the only place where tokens affect each other.


In [3]:
# Animation: one word travelling through the 16 chunks (4 layers each: three DeltaNet, one attention), then the pick.
html = r"""
<div id="__ID__" style="font:14px system-ui,sans-serif;max-width:700px">
 <div class="sent" style="display:flex;flex-wrap:wrap;gap:6px;min-height:34px;margin:0 0 8px"></div>
 <svg viewBox="0 0 680 236" width="100%" style="display:block"></svg>
 <div style="display:flex;align-items:center;gap:10px;margin-top:6px">
  <button class="play">Pause</button><button class="step">Step</button><span class="cap" style="opacity:.75;flex:1"></span>
 </div>
 <div style="font-size:12px;opacity:.7;margin-top:8px">
  <span style="display:inline-block;width:10px;height:10px;background:rgba(127,127,127,.6);border-radius:2px;margin-right:5px"></span>DeltaNet layer (fixed-size memory)
  &nbsp;&nbsp;<span style="display:inline-block;width:10px;height:10px;background:#7F77DD;border-radius:2px;margin-right:5px"></span>Attention layer (reads the cache)
 </div>
</div>
<script>
(function(){
 var root=document.getElementById('__ID__'), svg=root.querySelector('svg'), NS='http://www.w3.org/2000/svg';
 var words=['The','capital','of','France','is'], nexts=['Paris','.','It','is','a'], n=0, step=-1, playing=true, timer=null;
 function mk(t,a,p){var e=document.createElementNS(NS,t);for(var k in a)e.setAttribute(k,a[k]);(p||svg).appendChild(e);return e;}
 var pos=[], boxes=[];
 for(var i=0;i<16;i++){
  var x=24+(i%8)*80, y=14+Math.floor(i/8)*104;
  var r=mk('rect',{x:x,y:y,width:70,height:90,rx:6,fill:'rgba(127,127,127,.12)',stroke:'rgba(127,127,127,.45)','stroke-width':.8});
  var t=mk('text',{x:x+35,y:y+15,'text-anchor':'middle','font-size':11,fill:'currentColor'});t.textContent='Chunk '+(i+1);
  for(var k=0;k<4;k++) mk('rect',{x:x+10,y:y+24+k*15,width:50,height:11,rx:2,fill:k==3?'#7F77DD':'rgba(127,127,127,.6)'});
  boxes.push(r); pos.push([x+35,y+45]);
 }
 var dot=mk('circle',{r:9,cx:0,cy:0,fill:'#D85A30'}); dot.style.transition='transform .35s ease'; dot.style.transform='translate(8px,120px)';
 var cap=root.querySelector('.cap'), sent=root.querySelector('.sent');
 function draw(){
  sent.innerHTML='';
  words.concat(nexts.slice(0,step===17?n+1:n)).forEach(function(w,i){var s=document.createElement('span');s.textContent=w;
   s.style.cssText='padding:4px 10px;border-radius:8px;border:1px solid rgba(127,127,127,.4);'+(i>=words.length?'background:rgba(29,158,117,.3)':'background:rgba(127,127,127,.12)');sent.appendChild(s);});
  boxes.forEach(function(b,i){b.setAttribute('fill',i===step?'rgba(216,90,48,.5)':(step>i||step>=16)?'rgba(29,158,117,.3)':'rgba(127,127,127,.12)');});
  var c='The sentence is cut into tokens.';
  if(step>=0&&step<16){c='Chunk '+(step+1)+' of 16: four layers rewrite every token\'s 5,120 numbers.';var p=pos[step];dot.style.transform='translate('+p[0]+'px,'+p[1]+'px)';}
  if(step===16){c='Last step: score all 248,320 words the model knows.';dot.style.transform='translate(340px,226px)';}
  if(step===17){c='One word wins ('+nexts[n]+') and joins the sentence. Then everything runs again.';}
  cap.textContent=c;
 }
 function advance(){step++; if(step>17){step=-1;n=(n+1)%nexts.length;} draw();}
 function loop(){clearInterval(timer);if(playing)timer=setInterval(advance,450);root.querySelector('.play').textContent=playing?'Pause':'Play';}
 root.querySelector('.play').onclick=function(){playing=!playing;loop();};
 root.querySelector('.step').onclick=function(){playing=false;loop();advance();};
 draw(); loop();
})();
</script>
"""
show_html(html)

## 4. Building blocks


### 4.1 RMSNorm (zero-centered)

```
rms_norm(x) = x * rsqrt(mean(x^2) + eps) * (1 + w)         # note the (1 + w), w starts at 0
```

Computed in float32 in the reference. Forge builds `1 + w` into the stored weight (`ln1`, `ln2`, `qn`, `kn`).
Forge's `rms_hidden` (line 315) is a scale-free variant for FP16: it divides by `max|x|` first so squares stay in
[0, 1], because massive activations overflow and tiny layer-0 embeddings underflow.

The DeltaNet output uses a different, **gated** RMSNorm over each head's `dv = 128` values:

```
gated_norm(o, z) = ( o * rsqrt(mean(o^2) + eps) * w ) * silu(z)       # plain w here, not 1 + w
```


In [4]:
def rms_norm(x, w, eps=1e-6):
    """Zero-centered RMSNorm: the stored weight w starts at 0 and the scale is (1 + w)."""
    return x / np.sqrt((x ** 2).mean(-1, keepdims=True) + eps) * (1 + w)

def silu(u):
    return u / (1 + np.exp(-u))

def gated_rms_norm(o, z, w, eps=1e-6):
    """DeltaNet output norm: plain scale w, then multiplied by silu of the gate z."""
    return o / np.sqrt((o ** 2).mean(-1, keepdims=True) + eps) * w * silu(z)

rng = np.random.default_rng(0)
x = rng.normal(size=(3, 8)) * np.array([0.01, 1.0, 50.0])[:, None]       # very different sizes per row
y = rms_norm(x, np.zeros(8))
print("row size before:", np.sqrt((x ** 2).mean(-1)).round(3))
print("row size after: ", np.sqrt((y ** 2).mean(-1)).round(3), " <- all about 1 when w = 0")

row size before: [7.0000e-03 1.1200e+00 3.5805e+01]
row size after:  [0.989 1.    1.   ]  <- all about 1 when w = 0


### 4.2 Full attention (16 of 64 layers)

Learned matrices per layer: `q_proj: H -> nh*hd*2` (query and an output gate), `k_proj: H -> nkv*hd`,
`v_proj: H -> nkv*hd`, `o_proj: nh*hd -> H`. No biases.

```
[q, gate] = split(x_n * W_q)              # each (T, 24, 256); gate has the same shape as q
q = rms_norm_head(q)                      # q_norm over the 256-dim head, with its own (1 + w)
k = rms_norm_head(x_n * W_k)              # (T, 4, 256), k_norm likewise
v = x_n * W_v                             # (T, 4, 256)
q, k = rope(q, k)                         # only the first 64 of 256 dims are rotated (partial RoPE)
K_cache, V_cache <- append(k, v)          # (4, positions, 256) each
scores = (q * K_cache^T) / sqrt(256)      # divide by 16; each KV head serves 6 query heads
scores = scores + causal_mask
p = softmax(scores)                       # float32 in the reference
o = p * V_cache                           # (T, 24, 256)
o = o ⊙ σ(gate)                           # output gate, applied before the output projection
out = o_flat * W_o                        # (T, 6144) -> (T, 5120)
```

Rotary embedding on the first `rot = 64` dims (reference `apply_rotary_pos_emb`, line 674):

```
rotate_half(a) = concat(-a[32:64], a[0:32])
q_rot = q[:64] ⊙ cos + rotate_half(q[:64]) ⊙ sin ; the other 192 dims pass through unchanged
```

`cos` and `sin` are computed on the host from the position (theta = 10^7, interleaved mRoPE) and passed in as
inputs `cos`, `sin` of shape `(T, 64)`.

Cache cost per token, all 16 attention layers: `16 * 4 * 256 * 2 = 32,768` numbers. At 8 bits plus a 16-bit scale per
token and head that is 32.25 KiB (`docs/RELEASE_NOTES_v0.2.1.md`; `docs/KV_CACHE_V8.md` describes the older
FP16-key format).


In [5]:
# Full attention at toy size, same structure as the real layer:
# query + output gate in one projection, norm on each head, partial RoPE, grouped KV heads, 1/sqrt(hd) scaling.
def rotate_half(a):
    h = a.shape[-1] // 2
    return np.concatenate([-a[..., h:], a[..., :h]], -1)

def rope(t, pos, rot, theta=1e7):
    """Rotate only the first `rot` dims of each head; the rest pass through. t: (T, heads, hd)."""
    inv = theta ** (-np.arange(0, rot, 2) / rot)
    ang = pos[:, None] * inv[None, :]
    cos, sin = np.cos(np.concatenate([ang, ang], -1))[:, None, :], np.sin(np.concatenate([ang, ang], -1))[:, None, :]
    r, rest = t[..., :rot], t[..., rot:]
    return np.concatenate([r * cos + rotate_half(r) * sin, rest], -1)

def softmax(s):
    s = s - s.max(-1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(-1, keepdims=True)

class ToyAttention:
    def __init__(self, Hs=32, nh_=4, nkv_=2, hd_=16, rot_=4, seed=1):
        r = np.random.default_rng(seed)
        self.nh, self.nkv, self.hd, self.rot, self.grp = nh_, nkv_, hd_, rot_, nh_ // nkv_
        self.Wq = r.normal(size=(Hs, nh_ * hd_ * 2)) * 0.2          # query AND gate
        self.Wk = r.normal(size=(Hs, nkv_ * hd_)) * 0.2
        self.Wv = r.normal(size=(Hs, nkv_ * hd_)) * 0.2
        self.Wo = r.normal(size=(nh_ * hd_, Hs)) * 0.2
        self.qn, self.kn = r.normal(size=hd_) * 0.1, r.normal(size=hd_) * 0.1

    def project(self, x, pos):
        T = len(x)
        qg = (x @ self.Wq).reshape(T, self.nh, 2 * self.hd)
        q, gate = qg[..., :self.hd], qg[..., self.hd:]
        q = rms_norm(q, self.qn)
        k = rms_norm((x @ self.Wk).reshape(T, self.nkv, self.hd), self.kn)
        v = (x @ self.Wv).reshape(T, self.nkv, self.hd)
        return rope(q, pos, self.rot), gate, rope(k, pos, self.rot), v

    def attend(self, q, gate, K, V, qpos, kpos):
        """q: (Tq, nh, hd); K, V: (Tk, nkv, hd). Causal: a query sees keys at positions <= its own."""
        out = np.zeros((len(q), self.nh, self.hd))
        for h in range(self.nh):
            kv = h // self.grp                                          # 6 query heads share one KV head in the real model
            s = q[:, h] @ K[:, kv].T / np.sqrt(self.hd)
            s = np.where(kpos[None, :] <= qpos[:, None], s, -1e30)
            out[:, h] = softmax(s) @ V[:, kv]
        out = out * (1 / (1 + np.exp(-gate)))                           # sigmoid output gate
        return out.reshape(len(q), -1) @ self.Wo

    def full(self, x):
        pos = np.arange(len(x))
        q, g, k, v = self.project(x, pos)
        return self.attend(q, g, k, v, pos, pos)

att = ToyAttention()
x = np.random.default_rng(2).normal(size=(6, 32))
y = att.full(x)

# 1) causality: changing the LAST token must not change the earlier outputs
x2 = x.copy(); x2[-1] += 5.0
print("earlier outputs unchanged when the last token changes:", np.allclose(att.full(x2)[:-1], y[:-1]))

# 2) cache equivalence: feeding tokens one at a time with a growing KV cache gives the same outputs
Kc, Vc, kp, outs = np.zeros((0, att.nkv, att.hd)), np.zeros((0, att.nkv, att.hd)), np.zeros(0, int), []
for t in range(len(x)):
    pos = np.array([t])
    q, g, k, v = att.project(x[t:t + 1], pos)
    Kc, Vc, kp = np.concatenate([Kc, k]), np.concatenate([Vc, v]), np.concatenate([kp, pos])
    outs.append(att.attend(q, g, Kc, Vc, pos, kp)[0])
print("one-token-at-a-time with a KV cache matches the full pass:", np.allclose(np.array(outs), y))
print("cache after 6 tokens holds", Kc.shape, "keys and", Vc.shape, "values: it grows by one row per token")

earlier outputs unchanged when the last token changes: True
one-token-at-a-time with a KV cache matches the full pass: True
cache after 6 tokens holds (6, 2, 16) keys and (6, 2, 16) values: it grows by one row per token


In [6]:
# Animation: attention. Each token in turn asks a question and reads a weighted blend of earlier tokens (toy numbers).
html = r"""
<div id="__ID__" style="font:14px system-ui,sans-serif;max-width:700px">
 <div class="q" style="display:flex;gap:8px;flex-wrap:wrap;margin-bottom:12px"></div>
 <div class="rows"></div>
 <div class="out" style="margin-top:10px;padding:9px 12px;border-radius:8px;background:rgba(127,127,127,.12)"></div>
 <div style="display:flex;gap:10px;margin-top:10px"><button class="play">Pause</button></div>
 <div style="opacity:.7;font-size:12px;margin-top:8px">weight = softmax( (query . key) / sqrt(d) ). A token cannot see tokens after it. Toy numbers with 2 values per vector.</div>
</div>
<script>
(function(){
 var root=document.getElementById('__ID__'), $=function(c){return root.querySelector('.'+c);};
 var W=['The','capital','of','France','is'];
 var K=[[.1,.2],[1,.3],[.1,.1],[.9,1.1],[.4,.2]], V=[[0,.1],[.8,.2],[.1,0],[.2,.9],[.3,.3]], Q=[[.3,.2],[.8,.4],[.2,.3],[.9,1],[.9,1.2]];
 var qi=4, playing=true, timer=null, bars=[], pcts=[];
 var qd=$('q'), rows=$('rows');
 W.forEach(function(w,i){var b=document.createElement('button');b.textContent=w;b.onclick=function(){playing=false;loop();qi=i;draw();};qd.appendChild(b);});
 W.forEach(function(w,i){var r=document.createElement('div');r.style.cssText='display:grid;grid-template-columns:80px 1fr 54px;gap:10px;align-items:center;margin-bottom:6px;transition:opacity .4s';
  r.innerHTML='<span>'+w+'</span><div style="height:14px;background:rgba(127,127,127,.15);border-radius:4px;overflow:hidden"><div style="height:100%;width:0;background:#7F77DD;transition:width .6s"></div></div><span style="text-align:right;opacity:.8"></span>';
  rows.appendChild(r);bars.push(r);});
 function draw(){
  [].forEach.call(qd.children,function(b,i){b.style.cssText=i===qi?'font-weight:600;outline:2px solid #D85A30':'';});
  var d=Math.sqrt(2),sc=[],mx=-1e9;
  for(var j=0;j<5;j++){sc[j]=j<=qi?(Q[qi][0]*K[j][0]+Q[qi][1]*K[j][1])/d:null;if(sc[j]!==null&&sc[j]>mx)mx=sc[j];}
  var ex=sc.map(function(s){return s===null?0:Math.exp(s-mx);}), sum=ex.reduce(function(a,b){return a+b;},0), o=[0,0];
  for(var j=0;j<5;j++){var w=ex[j]/sum;o[0]+=w*V[j][0];o[1]+=w*V[j][1];
   bars[j].style.opacity=j<=qi?1:.3; bars[j].children[1].firstChild.style.width=(w*100)+'%'; bars[j].children[2].textContent=j<=qi?(w*100).toFixed(0)+'%':'hidden';}
  $('out').innerHTML='<b>'+W[qi]+'</b> reads the blend <span style="font-family:ui-monospace,monospace">['+o[0].toFixed(2)+', '+o[1].toFixed(2)+']</span>';
 }
 function loop(){clearInterval(timer);if(playing)timer=setInterval(function(){qi=(qi+1)%5;draw();},1800);$('play').textContent=playing?'Pause':'Play';}
 $('play').onclick=function(){playing=!playing;loop();};
 draw(); loop();
})();
</script>
"""
show_html(html)

### 4.3 Gated DeltaNet (48 of 64 layers)

Learned per layer: `in_proj_qkv: H -> 10,240`, `in_proj_z: H -> 6,144`, `in_proj_b: H -> 48`, `in_proj_a: H -> 48`,
a depthwise causal `conv1d` (kernel 4, over the 10,240 channels), `A_log (48)`, `dt_bias (48)`,
`norm.weight (128)`, `out_proj: 6,144 -> H`.

**Per token, step by step** (reference `Qwen3_5GatedDeltaNet.forward`, line 550):

```
mixed = silu( causal_depthwise_conv4( x_n * W_qkv ) )     # (T, 10240); needs the previous 3 rows as state
q, k, v = split(mixed, [2048, 2048, 6144])                # q, k: 16 heads of 128; v: 48 heads of 128
q, k   = repeat each key head 3 times  ->  48 heads       # 16 key heads feed 48 value heads
z      = x_n * W_z                                        # (T, 48, 128), only used for the output gate
beta   = σ( x_n * W_b )                                   # (T, 48)   write strength, in (0, 1)
g      = -exp(A_log) * softplus( x_n * W_a + dt_bias )    # (T, 48)   log of the decay, always <= 0
q = l2norm(q) ;  k = l2norm(k)                            # unit length per head
q = q / sqrt(128)
```

**The recurrence** (reference `torch_recurrent_gated_delta_rule`, line 438). One state matrix `S` per value head,
shape `(dk, dv) = (128, 128)`:

```
S  = exp(g_t) * S                          # decay: forget a little (g <= 0, so the factor is in (0, 1])
m  = S^T * k_t                             # what memory currently returns for key k_t   (dv numbers)
d  = beta_t * (v_t - m)                    # the correction: the difference between wanted and stored
S  = S + k_t ⊗ d                           # rank-1 write: outer product (dk x dv)
o_t = S^T * q_t                            # read with the query
```

This is the "delta rule": a write replaces what the same key returned before, instead of piling on.
`exp(g)` is a per-head forgetting factor, which is the "gated" part. `S` never grows with sequence length.

**Output:**

```
o = gated_norm(o, z)                       # per head, 128 values; see 4.1
out = o_flat * W_out                       # (T, 6144) -> (T, 5120)
```

State per layer: `S` of `48 * 128 * 128 = 786,432` numbers plus the last 3 conv rows (`3 * 10,240`).
Across 48 layers about 38 million numbers, the same at any context length.


#### 4.3.1 Chunked form (what Forge actually computes)

The loop above is serial in time. The ANE graph processes blocks of 8 rows at once. Let `C = 8` rows in a block,
`g` and `beta` for the block, and `S` the state before it. All shapes are per value head.

```
cum    = cumsum(g)                                         # (C,)  running decay inside the block
pair_ij = exp(cum_i - cum_j)   for j <= i, else 0          # decay between row j and row i
kb = k ⊙ beta ;  vb = v ⊙ beta
N  = (kb * k^T) ⊙ pair ⊙ strict_lower_mask                 # (C, C) interaction between rows of the block
solve (I + N) X = [ vb , kb ⊙ exp(cum) ]                   # (C, dv + dk)
u  = X[:, :dv] ;  wk = X[:, dv:]
v_new = u - wk * S                                         # the part of v the old state does not explain
o   = (q ⊙ exp(cum)) * S  +  ((q * k^T) ⊙ pair) * v_new    # old memory + the block's own writes
S'  = S * exp(cum_last) + (k ⊙ exp(cum_last - cum))^T * v_new
```

This is the WY / UT form of the delta rule (Songlin Yang, "DeltaNet Explained, Part II", linked in the builder).
Forge code: `GDNW.verify` lines 464-482, `GDNW.prefill` lines 484-518.

Checked: `python docs/ARCHITECTURE_CHEATSHEET_check.py` runs both forms on random data. Outputs and final state
agree with the token-by-token recurrence to about 1e-16, and the Neumann-product inverse equals `inv(I + N)`.
This verifies the equations as written here; it does not run the ANE graph.

`(I + N)` is unit lower-triangular, so `(I + N)^-1` is a finite sum. Forge computes it with the Neumann product
`(I - N)(I + N^2)(I + N^4)...` (`inv_unit_lower`, line 380) because the ANE compiler rejects row-by-row substitution
and matmul chains between computed tensors. With `GDN_FAST=0` it falls back to `fwd_sub` (line 369).


In [7]:
# DeltaNet (one value head): token by token versus the chunked form Forge computes.
def recurrent(S0, q, k, v, beta, g):
    """Reference recurrence (transformers torch_recurrent_gated_delta_rule). S has shape (dk, dv)."""
    S, out = S0.copy(), []
    for t in range(len(q)):
        S = S * np.exp(g[t])                         # decay
        d = beta[t] * (v[t] - S.T @ k[t])            # correction: wanted minus what memory returns for this key
        S = S + np.outer(k[t], d)                    # rank-1 write
        out.append(S.T @ q[t])                       # read
    return np.array(out), S

def chunked(S0, q, k, v, beta, g):
    """One block of C rows at once (GDNW.verify / prefill in the builder)."""
    C = len(q)
    lower, strict = np.tril(np.ones((C, C))), np.tril(np.ones((C, C)), -1)
    cum = np.cumsum(g)
    pair = np.exp(cum[:, None] - cum[None, :]) * lower
    kb, vb = k * beta[:, None], v * beta[:, None]
    N = (kb @ k.T) * pair * strict
    X = np.linalg.solve(np.eye(C) + N, np.concatenate([vb, kb * np.exp(cum)[:, None]], 1))
    u, wk = X[:, :v.shape[1]], X[:, v.shape[1]:]
    vn = u - wk @ S0
    o = (q * np.exp(cum)[:, None]) @ S0 + ((q @ k.T) * pair) @ vn
    S1 = S0 * np.exp(cum[-1]) + (k * np.exp(cum[-1] - cum)[:, None]).T @ vn
    return o, S1

def inv_unit_lower(n, rows):
    """(I + N)^-1 for strictly lower-triangular N as (I - N)(I + N^2)(I + N^4)... (Forge: inv_unit_lower)."""
    eye, t, p, step = np.eye(rows), np.eye(rows) - n, n, 2
    while step < rows:
        p = p @ p
        t = t @ (eye + p)
        step *= 2
    return t

rng = np.random.default_rng(0)
T, dk_, dv_ = 24, 5, 6                                   # three blocks of 8 rows
q, k = rng.normal(size=(T, dk_)), rng.normal(size=(T, dk_))
q /= np.linalg.norm(q, axis=1, keepdims=True); k /= np.linalg.norm(k, axis=1, keepdims=True)   # unit length, as in the layer
v = rng.normal(size=(T, dv_))
beta = 1 / (1 + np.exp(-rng.normal(size=T)))             # sigmoid: write strength in (0, 1)
g = -np.abs(rng.normal(size=T)) * 0.5                    # log decay <= 0
S0 = rng.normal(size=(dk_, dv_)) * 0.3

o_ref, S_ref = recurrent(S0, q, k, v, beta, g)
S, outs = S0, []
for b in range(0, T, 8):                                 # carry the state from block to block
    o, S = chunked(S, q[b:b + 8], k[b:b + 8], v[b:b + 8], beta[b:b + 8], g[b:b + 8])
    outs.append(o)
print("chunked outputs match token-by-token:", np.allclose(np.concatenate(outs), o_ref))
print("final state matches:                 ", np.allclose(S, S_ref))

cum = np.cumsum(g[:8]); pair = np.exp(cum[:, None] - cum[None, :]) * np.tril(np.ones((8, 8)))
N = ((k[:8] * beta[:8, None]) @ k[:8].T) * pair * np.tril(np.ones((8, 8)), -1)
print("Neumann-product inverse equals inv(I + N):", np.allclose(inv_unit_lower(N, 8), np.linalg.inv(np.eye(8) + N)))

# The delta rule overwrites: writing a new value under the SAME key replaces the old one.
S = np.zeros((2, 2)); key = np.array([1.0, 0.0])
for val in ([1.0, 0.2], [-0.5, 0.8]):
    S = S + np.outer(key, 1.0 * (np.array(val) - S.T @ key))      # beta = 1, no decay
    print("asking with the key returns", (S.T @ key).round(2))
print("DeltaNet memory shape stays", S.shape, "however many tokens arrive")

chunked outputs match token-by-token: True
final state matches:                  True
Neumann-product inverse equals inv(I + N): True
asking with the key returns [1.  0.2]
asking with the key returns [-0.5  0.8]
DeltaNet memory shape stays (2, 2) however many tokens arrive


In [8]:
# Animation: DeltaNet memory. A tiny 2 x 2 grid is updated token by token (toy numbers, no decay, write strength 1).
html = r"""
<div id="__ID__" style="font:14px system-ui,sans-serif;max-width:700px">
 <div style="display:flex;gap:24px;flex-wrap:wrap;align-items:flex-start">
  <div style="flex:1;min-width:260px"><div style="opacity:.7;font-size:12px;margin-bottom:6px">Tokens arriving (key, value)</div><div class="seq" style="display:flex;flex-direction:column;gap:6px"></div></div>
  <div><div style="opacity:.7;font-size:12px;margin-bottom:6px">The memory: always a 2 x 2 grid</div>
   <div class="mat" style="display:grid;grid-template-columns:repeat(2,76px);gap:6px"></div>
   <div class="rec" style="margin-top:10px;font-size:13px;max-width:240px"></div></div>
 </div>
 <div style="display:flex;gap:10px;margin-top:12px"><button class="play">Pause</button><button class="next">Next token</button><button class="reset">Reset</button></div>
</div>
<script>
(function(){
 var root=document.getElementById('__ID__'), $=function(c){return root.querySelector('.'+c);};
 var TK=[{n:'token 1',k:[1,0],v:[1,.2]},{n:'token 2',k:[0,1],v:[.1,1]},{n:'token 3',k:[.707,.707],v:[.9,.9]},{n:'token 4 (same key as token 1)',k:[1,0],v:[-.5,.8]}];
 var S,t,playing=true,timer=null;
 function mv(M,x){return [M[0][0]*x[0]+M[0][1]*x[1],M[1][0]*x[0]+M[1][1]*x[1]];}
 function reset(){S=[[0,0],[0,0]];t=0;draw(-1);}
 function step(){ if(t>=TK.length){ reset(); return; }
  var k=TK[t].k,v=TK[t].v,Sk=mv(S,k),e=[v[0]-Sk[0],v[1]-Sk[1]];
  S=[[S[0][0]+e[0]*k[0],S[0][1]+e[0]*k[1]],[S[1][0]+e[1]*k[0],S[1][1]+e[1]*k[1]]]; t++; draw(t-1); }
 function draw(cur){
  var sq=$('seq'); sq.innerHTML='';
  TK.forEach(function(x,i){var d=document.createElement('div');
   d.style.cssText='padding:7px 10px;border-radius:8px;border:1px solid rgba(127,127,127,.4);transition:all .3s;font-size:13px;'+(i===cur?'background:rgba(216,90,48,.4)':i<t?'background:rgba(29,158,117,.3)':'opacity:.5');
   d.innerHTML=x.n+'<br><span style="font-family:ui-monospace,monospace">k=['+x.k.map(function(n){return n.toFixed(2);}).join(', ')+'] v=['+x.v.map(function(n){return n.toFixed(1);}).join(', ')+']</span>';sq.appendChild(d);});
  var m=$('mat'); m.innerHTML='';
  [S[0][0],S[0][1],S[1][0],S[1][1]].forEach(function(n){var c=document.createElement('div');var a=Math.min(1,Math.abs(n));
   c.style.cssText='height:58px;border-radius:8px;display:flex;align-items:center;justify-content:center;font-weight:600;border:1px solid rgba(127,127,127,.4);transition:background .5s;background:'+(n>=0?'rgba(127,119,221,':'rgba(216,90,48,')+(.1+.6*a)+')';c.textContent=n.toFixed(2);m.appendChild(c);});
  var r=mv(S,TK[0].k);
  $('rec').innerHTML=t?('Ask with token 1\'s key: <b>['+r.map(function(n){return n.toFixed(2);}).join(', ')+']</b>'+(t===4?'<br>Token 4 reused that key, so its value replaced token 1\'s. The grid did not grow.':'')):'Empty memory.';
 }
 function loop(){clearInterval(timer);if(playing)timer=setInterval(step,1700);$('play').textContent=playing?'Pause':'Play';}
 $('play').onclick=function(){playing=!playing;loop();};
 $('next').onclick=function(){playing=false;loop();step();};
 $('reset').onclick=function(){reset();};
 reset(); loop();
})();
</script>
"""
show_html(html)

### 4.4 MLP (all 64 layers, about 17 billion weights)

```
a   = silu( x_n * W_gate ) ⊙ ( x_n * W_up )     # (T, 5120) -> (T, 17408), two paths
out = a * W_down                                 # (T, 17408) -> (T, 5120)
```

Three matrices of `5120 x 17408 = 89,128,960` weights each, so 267,386,880 per layer. Source: reference line 835;
Forge `LayerW.mlp` (line 794). In Forge the MLP input and the middle activation are optionally rotated by an online
block-Hadamard matrix (`Hadamard`, block size 1024) before the quantized matmuls; seeds come from the export metadata.


In [9]:
def mlp(x, Wg, Wu, Wd):
    """SwiGLU: down( silu(x Wg) * (x Wu) )."""
    return (silu(x @ Wg) * (x @ Wu)) @ Wd

rng = np.random.default_rng(3)
Hs, Fs = 16, 56                                          # toy sizes; the real ones are 5120 and 17408
Wg, Wu, Wd = rng.normal(size=(Hs, Fs)) * .2, rng.normal(size=(Hs, Fs)) * .2, rng.normal(size=(Fs, Hs)) * .2
x = rng.normal(size=(4, Hs))
print("toy MLP:", x.shape, "->", (x @ Wg).shape, "inside ->", mlp(x, Wg, Wu, Wd).shape)
print(f"real model: {H} -> {F} -> {H}, {3 * H * F:,} weights per layer, {3 * H * F * len(layer_types) / 1e9:.1f} billion in all")

toy MLP: (4, 16) -> (4, 56) inside -> (4, 16)
real model: 5120 -> 17408 -> 5120, 267,386,880 weights per layer, 17.1 billion in all


### 4.5 Output head

`logits = RMSNorm_final(x_last) * W_head`, `W_head` is `5120 x 248,320`. Forge compiles it as `head_T8.aimodel` in
8 row parts with a 4-bit lookup table (`Head`, line 902). Sampling happens on the CPU in `qwen38_server.py`.


## 5. What number type each piece uses in Forge

| Piece | Stored as | Reconstructed as |
| --- | --- | --- |
| All MLP matrices (gate, up, down), release 0.2 | 6-bit index per pair of weights (64 entries of 2) = 3 bits per weight | `W[2g+t, j] = s[2g+t] * L[idx[g, j], t]`, FP16 codebook `L (64, 2)`, FP16 scale `s` per output row (`docs/QUANTIZATION.md`, `vector 2x64`) |
| Token-mixer projections (DeltaNet `in_proj_qkv`, `in_proj_z`, `out_proj`; attention `q_proj`, `o_proj`) | Lookup tables of 2 to 4 bits. In the first release's plan: vector 2x16 for layers 0-23, scalar 16-entry (LUT4) for layers 24-63. Release 0.2's exact mixer plan: **(unverified)** | same `QConv` LUT path, with per-row scale |
| Attention `k_proj`, `v_proj` | INT8 with per-channel scales (compile-time constant) | `QConv` path with `w8` (line 232) |
| DeltaNet `in_proj_a`, `in_proj_b` | dense FP16 | as is (line 411-412) |
| Mixer error correction | FP16 rank-64 pair `a @ (b @ x)` on selected mixer matrices | added to the conv output (`lr_a`, `lr_b`) |
| Output head | 4-bit lookup table (LUT4) in 8 row parts | `Head` (line 902) |
| Norm weights, conv taps, `A_log`, `dt_bias` | FP16 | as is |
| Activations between layers | FP16 | |
| Attention K and V cache | INT8 + FP16 scale per token and head (`kv8`) | `code * scale` |
| Attention scores / probabilities | INT8 scores (step 1/4), FP8 softmax values (scale 1/64) on M6 | form list `s8,s8b,sm8,pvf8` |
| DeltaNet state `S` | fixed-size tensor `(nv, dk, dv)` | reference uses float32; the ANE build's exact precision is **(unverified)** |

M5 chips reject the FP8 forms, which is why release 0.2.1 is M6 only.


In [10]:
# Rebuilding weights from a vector lookup table ("vector 2x64", release 0.2 MLP format).
rng = np.random.default_rng(4)
Cout, Cin = 8, 6
codebook = rng.normal(size=(64, 2)).astype(np.float16)           # L: 64 entries of 2 numbers, shared by the whole matrix
idx = rng.integers(0, 64, size=(Cout // 2, Cin))                  # I: one 6-bit index per PAIR of weights
scale = (rng.random(Cout) * 3 + 0.5).astype(np.float16)           # s: one scale per output row

W_hat = np.zeros((Cout, Cin), np.float32)
for g_ in range(Cout // 2):
    for t in range(2):
        W_hat[2 * g_ + t, :] = scale[2 * g_ + t] * codebook[idx[g_], t]   # W[2g+t, j] = s[2g+t] * L[I[g, j], t]
print("rebuilt weight matrix:", W_hat.shape, "from", idx.size, "six-bit indices")

def packed_bytes(cout, cin):
    """3 bits per weight, plus a 256-byte codebook and 2 bytes per output row (docs/QUANTIZATION.md)."""
    return 3 * cout * cin / 8 + 256 + 2 * cout

real = packed_bytes(F, H)
print(f"one real MLP matrix {F} x {H}: {real / 1e6:.1f} MB packed vs {F * H * 2 / 1e6:.1f} MB in 16-bit "
      f"({F * H * 2 / real:.1f}x smaller)")

rebuilt weight matrix: (8, 6) from 24 six-bit indices
one real MLP matrix 17408 x 5120: 33.5 MB packed vs 178.3 MB in 16-bit (5.3x smaller)


## 6. Why Forge's graph looks different from the reference

| Difference | Reason | Where |
| --- | --- | --- |
| Tensors are channel-major `(1, C, 1, T)` and linears are 1x1 convs | The ANE prefers this layout | `QConv`, `Entry` |
| Fixed-shape entry points per context size (`v8_8k`, `p64_16k`, ...) | The ANE compiles fixed shapes; the runtime picks the smallest entry that fits and copies the cache | `Entry`, `docs/SPECULATIVE_DECODING.md` section "How ANE context expansion is implemented" |
| Verify entries use `T = 8` rows, prefill 64 rows (32 above 96K) | One verify call checks the anchor plus 7 draft tokens | `P, C_SUB = 8, 8` (line 44), `TPS`, `TPS_LONG` |
| DeltaNet "lazy commit": `pend`, `commit`, `commit_last` | DeltaNet state cannot be rewound, so a verify call stores its candidate rows as pending and the next call commits only the accepted prefix | `GDNW.commit_pending` (448) |
| History attention is tiled (`ATT_BLOCK` 2048 / 4096) with one global max | The ANE compiler fails on a single softmax wider than 65,536 | `AttnW.forward`, `kv_len` (807) |
| `GDN_SQ = 16`, `GDN_SV = 64` scaling around the DeltaNet core | Keeps `q * S` out of FP16 subnormals; undone in `finish` | lines 342, 445, 457 |
| Custom `softplus`, `silu` via tanh | The ANE's FP16 softplus returns 0 above about 11; native silu has about 1e-3 absolute error near 0 | lines 327, 357 |
| Online Hadamard rotations on quantized matmul inputs | Spreads outliers so 3-bit weights lose less accuracy | `Hadamard` (264), `mixer_rotation` (297) |
| Drafter reuses the target's output head | Saves 0.86 GB of Neural Engine memory | release notes 0.2.1 |


## 7. Interface of one compiled entry (one chunk of 4 layers)

From `Entry.input_names` / `output_names` (around line 838):

Inputs (in order):

```
x            (1, 5120, 1, T)        hidden states entering the chunk, FP16
cos, sin     (T, 64)                RoPE tables for these positions
mask         (1, ctx)               0 for visible history rows, large negative otherwise
conv_sel     (3, P+3)               picks the 3 conv rows to carry in
commit       (1, P, 1)              which pending rows to commit this call
commit_last  (1, P, 1)              the last committed row's selector
conv_sel_out, valid   prefill only  row selector and 0/1 validity of padded rows
per DeltaNet layer j:  conv_j (P+3, 10240), rec_j (48, 128, 128), pend_j (48, 3P+1, 128)
per attention layer j: k_j, v_j (+ ks_j, vs_j scales for INT8)   (4, ctx, 256)
```

Outputs:

```
y                       hidden states leaving the chunk
tap<l>                  features for the drafter, layers in TAPS only
per DeltaNet layer:     conv_j_out, rec_j_out, pend_j_out
per attention layer:    k_j_new, v_j_new (the runtime writes accepted rows into the cache)
```

A forward pass is 16 such chunk calls in order, then the head.


## 8. Speculative decoding in one paragraph

After prefill the target picks an anchor token. DFlash2 proposes 7 following tokens. One target call evaluates
`[anchor, d1 .. d7]` as `T = 8` rows. The server accepts drafts in order while each matches what the target would
pick (under sampling: accepted with the target's probability of that token) and takes the target's own token at the
first miss. If all 7 pass, the 8th row yields a bonus token. Only the accepted prefix is committed (`accept(k)`),
which is why DeltaNet needs the lazy-commit machinery. The drafter has its own 2,048-row ring and is unaffected by
the target's context size. Source: `docs/SPECULATIVE_DECODING.md`, `scripts/qwen38_server.py`.


In [11]:
# Animation: speculative decoding. The drafter guesses 7 words, the big model checks them in ONE pass.
# How many guesses are accepted per cycle is drawn from your server's real counts (51, 51, 41, 28, 24, 14, 12, 21 for 0..7).
html = r"""
<div id="__ID__" style="font:14px system-ui,sans-serif;max-width:700px">
 <div style="opacity:.7;font-size:12px;margin-bottom:4px">Written so far</div>
 <div class="out" style="min-height:34px;display:flex;flex-wrap:wrap;gap:5px;margin-bottom:12px"></div>
 <div style="opacity:.7;font-size:12px;margin-bottom:4px">The drafter's 7 guesses</div>
 <div class="guess" style="min-height:38px;display:flex;flex-wrap:wrap;gap:6px;margin-bottom:8px"></div>
 <div class="msg" style="min-height:22px;margin-bottom:10px"></div>
 <div style="display:flex;gap:10px;align-items:center;margin-bottom:12px">
  <button class="play">Pause</button><button class="next">Next cycle</button><button class="reset">Reset</button>
 </div>
 <div style="display:flex;gap:10px;flex-wrap:wrap">
  <div style="flex:1;min-width:130px;padding:10px 12px;border-radius:8px;background:rgba(127,127,127,.12)"><div style="opacity:.7;font-size:12px">Big-model passes</div><div class="passes" style="font-size:22px;font-weight:600">0</div></div>
  <div style="flex:1;min-width:130px;padding:10px 12px;border-radius:8px;background:rgba(127,127,127,.12)"><div style="opacity:.7;font-size:12px">Words written</div><div class="words" style="font-size:22px;font-weight:600">0</div></div>
  <div style="flex:1;min-width:130px;padding:10px 12px;border-radius:8px;background:rgba(127,127,127,.12)"><div style="opacity:.7;font-size:12px">Words per pass</div><div class="avg" style="font-size:22px;font-weight:600">-</div></div>
 </div>
 <div style="opacity:.7;font-size:12px;margin-top:8px">Without a drafter every word needs its own pass (1.0 words per pass).</div>
</div>
<script>
(function(){
 var root=document.getElementById('__ID__'), $=function(c){return root.querySelector('.'+c);};
 var corpus='The capital of France is Paris , which is also the largest city in the country . It sits on the river Seine and is known for the Louvre and the Eiffel Tower .'.split(' ');
 var decoys=['London','a','very','was','blue','of','and','Rome','city','big'];
 var hist=[51,51,41,28,24,14,12,21], tot=hist.reduce(function(a,b){return a+b;},0);
 var START=5, pos=START, passes=0, words=0, playing=true, busy=false, gen=0;
 var sleep=function(ms){return new Promise(function(r){setTimeout(r,ms);});};
 function pick(){var r=Math.random()*tot;for(var k=0;k<8;k++){r-=hist[k];if(r<0)return k;}return 7;}
 function chip(t,css){var s=document.createElement('span');s.textContent=t;s.style.cssText='padding:5px 11px;border-radius:8px;border:1px solid rgba(127,127,127,.4);transition:all .25s;'+(css||'');return s;}
 function show(){$('passes').textContent=passes;$('words').textContent=words;$('avg').textContent=passes?(words/passes).toFixed(1):'-';}
 function resetAll(){gen++;pos=START;passes=0;words=0;busy=false;$('out').innerHTML='';corpus.slice(0,START).forEach(function(w){$('out').appendChild(chip(w,'background:rgba(127,127,127,.12)'));});$('guess').innerHTML='';$('msg').textContent='';show();}
 async function cycle(){
  if(busy)return; busy=true; var my=gen;
  if(pos+9>corpus.length) { resetAll(); busy=true; my=gen; }
  var a=pick(), g=[];
  for(var i=0;i<7;i++){ if(i<a) g.push(corpus[pos+i]); else { var d; do{d=decoys[Math.floor(Math.random()*decoys.length)];}while(d===corpus[pos+i]); g.push(d);} }
  $('guess').innerHTML=''; $('msg').textContent='The small drafter guesses the next 7 words...';
  var els=g.map(function(w){var e=chip(w,'opacity:0;background:rgba(127,127,127,.12)');$('guess').appendChild(e);return e;});
  for(var i=0;i<7;i++){ await sleep(130); if(my!==gen)return; els[i].style.opacity=1; }
  await sleep(350); $('msg').textContent='The big model checks all 7 in a single pass...';
  for(var i=0;i<7;i++){
   await sleep(240); if(my!==gen)return;
   if(i<a){ els[i].style.background='rgba(29,158,117,.4)'; }
   else { els[i].style.background='rgba(216,90,48,.45)'; els[i].style.textDecoration='line-through'; for(var j=i+1;j<7;j++) els[j].style.opacity=.3; break; }
  }
  await sleep(300); if(my!==gen)return;
  var add=corpus.slice(pos,pos+a); add.push(corpus[pos+a]); if(a===7) add.push(corpus[pos+8]);
  passes++; words+=add.length;
  $('msg').textContent=a===0?'First guess wrong, but the big model still writes 1 correct word.':(a===7?'All 7 right, plus 1 bonus word: 8 words from one pass.':a+' guesses accepted, then the big model fixes the next one: '+add.length+' words from one pass.');
  add.forEach(function(w,i){$('out').appendChild(chip(w,'background:rgba(29,158,117,.3)'));});
  pos+=add.length; show(); busy=false;
 }
 async function run(){ while(playing){ await cycle(); await sleep(900);} }
 $('play').onclick=function(){playing=!playing;$('play').textContent=playing?'Pause':'Play';if(playing)run();};
 $('next').onclick=function(){playing=false;$('play').textContent='Play';cycle();};
 $('reset').onclick=function(){resetAll();};
 resetAll(); run();
})();
</script>
"""
show_html(html)

## 9. Extension recipes

These are my reading of the code, not tested changes. For each, the validation you need is in section 10.


### 9.1 Longer context (beyond the 192K ladder)

- The model's own limit is 262,144 positions, so 256K is the real ceiling without extending RoPE.
- Context entries come from the `--ctx` list of the builder (`main`, `--ctx 8192,...`) and `forge.py compile --ctx`.
  Add the new size to the ladder and to `TPS_LONG`, `PREFILL_LONG_FROM`, `ATT_BLOCK_LONG`, `ATT_BLOCK_PREFILL_LONG`
  (lines 45-63). `kv_len` (807) already lets entries above 65,536 use their full length through the tiled path.
- Cost: INT8 cache is 32.25 KiB per position, so 256K is about 8.1 GiB. With 16.5 GB of weights on the Neural Engine
  and about 2 GB of process memory, that is about 26.6 GB on a 32 GB Mac. Tight; swap would likely start **(unverified)**.
- The runtime also needs the entry in its manifest and in the cache-resize logic (`qwen38_coreai_model.py`).
- Prefill time rises quickly: the notes list 62 tok/s at 160K and 192K.


In [12]:
# Memory budget for a context length. The inputs are the measured figures quoted in the notes.
KV_KIB_PER_TOKEN = 32.25            # INT8 keys and values plus FP16 scales, all 16 attention layers
WEIGHTS_GB = 16.5                   # Neural Engine memory, all3 drafter (release notes 0.2.1)
PROCESS_GB = 2.0                    # Python process without the cache (about 1.5 to 2 GB at short context)
MAC_GB, OS_RESERVE_GB = 32, 6       # your Mac, and what macOS and a few apps need

def budget(ctx_tokens, kv_kib=KV_KIB_PER_TOKEN):
    cache_gb = ctx_tokens * kv_kib / 1024 / 1024
    total = WEIGHTS_GB + PROCESS_GB + cache_gb
    return cache_gb, total, MAC_GB - OS_RESERVE_GB - total

print(f"{'context':>8} {'cache GiB':>10} {'total GB':>9} {'spare GB':>9}   (spare below 0 means swapping)")
for k in (16, 32, 64, 128, 192, 256):
    c, t, s = budget(k * 1024)
    print(f"{k:>7}K {c:>10.2f} {t:>9.1f} {s:>9.1f}")
print("\nwith a 16-bit cache (64 KiB per token):")
for k in (128, 192):
    c, t, s = budget(k * 1024, 64)
    print(f"{k:>7}K {c:>10.2f} {t:>9.1f} {s:>9.1f}")

 context  cache GiB  total GB  spare GB   (spare below 0 means swapping)
     16K       0.50      19.0       7.0
     32K       1.01      19.5       6.5
     64K       2.02      20.5       5.5
    128K       4.03      22.5       3.5
    192K       6.05      24.5       1.5
    256K       8.06      26.6      -0.6

with a 16-bit cache (64 KiB per token):
    128K       8.00      26.5      -0.5
    192K      12.00      30.5      -4.5


In [13]:
# Animation: memory growth as a conversation gets longer. Attention cache grows; DeltaNet memory stays flat.
# Cache: 32.25 KiB per token for the 16 attention layers. DeltaNet: 37.7 million numbers, drawn at 16 bits (an assumption).
# "If all 64 layers were attention" is hypothetical: 4 times the cache.
html = r"""
<div id="__ID__" style="font:14px system-ui,sans-serif;max-width:700px">
 <svg viewBox="0 0 680 270" width="100%" style="display:block"></svg>
 <div style="display:flex;gap:10px;align-items:center;margin-top:6px"><button class="play">Replay</button><span class="read" style="opacity:.85"></span></div>
</div>
<script>
(function(){
 var root=document.getElementById('__ID__'), svg=root.querySelector('svg'), NS='http://www.w3.org/2000/svg';
 function mk(t,a){var e=document.createElementNS(NS,t);for(var k in a)e.setAttribute(k,a[k]);svg.appendChild(e);return e;}
 var X0=70,X1=650,Y0=230,Y1=20,TMAX=196608,GMAX=26;
 var sx=function(t){return X0+(X1-X0)*t/TMAX;}, sy=function(g){return Y0-(Y0-Y1)*g/GMAX;};
 mk('line',{x1:X0,y1:Y0,x2:X1,y2:Y0,stroke:'rgba(127,127,127,.6)'}); mk('line',{x1:X0,y1:Y0,x2:X0,y2:Y1,stroke:'rgba(127,127,127,.6)'});
 [0,8,16,24].forEach(function(g){mk('line',{x1:X0,y1:sy(g),x2:X1,y2:sy(g),stroke:'rgba(127,127,127,.18)'});var t=mk('text',{x:X0-8,y:sy(g)+4,'text-anchor':'end','font-size':11,fill:'currentColor'});t.textContent=g+' GiB';});
 [0,64,128,192].forEach(function(k){var t=mk('text',{x:sx(k*1024),y:Y0+16,'text-anchor':'middle','font-size':11,fill:'currentColor'});t.textContent=k+'K tokens';});
 var gib=function(t,mult){return t*32.25*mult/1024/1024;};
 var lines=[{m:4,c:'rgba(127,127,127,.7)',d:'4 3',label:'if all 64 layers were attention (hypothetical)'},{m:1,c:'#D85A30',d:'',label:'attention cache, 16 layers (real)'},{m:0,c:'#7F77DD',d:'',label:'DeltaNet memory, 48 layers (flat)'}];
 var paths=lines.map(function(l){return mk('polyline',{fill:'none',stroke:l.c,'stroke-width':2.5,'stroke-dasharray':l.d});});
 lines.forEach(function(l,i){var t=mk('text',{x:X0+8,y:34+i*15,'font-size':12,fill:l.c});t.textContent=l.label;});
 var dotv=mk('line',{y1:Y0,y2:Y1,stroke:'rgba(127,127,127,.5)','stroke-dasharray':'3 3'});
 var read=root.querySelector('.read'), t0=null, DUR=7000, raf=null;
 function val(l,t){return l.m===0?0.07:gib(t,l.m);}
 function draw(t){
  paths.forEach(function(p,i){var pts=[];for(var s=0;s<=t;s+=TMAX/80)pts.push(sx(s)+','+sy(val(lines[i],s)));pts.push(sx(t)+','+sy(val(lines[i],t)));p.setAttribute('points',pts.join(' '));});
  dotv.setAttribute('x1',sx(t));dotv.setAttribute('x2',sx(t));
  read.textContent=Math.round(t/1024)+'K tokens: attention cache '+gib(t,1).toFixed(2)+' GiB, DeltaNet memory about 0.07 GiB (flat)';
 }
 function frame(ts){ if(t0===null)t0=ts; var f=Math.min(1,(ts-t0)/DUR); draw(f*TMAX); if(f<1)raf=requestAnimationFrame(frame); }
 function play(){ cancelAnimationFrame(raf); t0=null; raf=requestAnimationFrame(frame); }
 root.querySelector('.play').onclick=play; play();
})();
</script>
"""
show_html(html)

### 9.2 Smaller KV cache (for more context in the same memory)

- Today K and V are INT8 with one FP16 scale per token and KV head. 4-bit keys or values would roughly halve the
  per-token cost.
- Touch points: `KV_INPUTS` and `INT8_LAYOUTS` (`scripts/qwen38_kv_cache.py`), the `quant8` / `dequant8` pairs and
  the scale handling in `AttnW.forward`, and the runtime code that writes accepted rows.
- Risk: attention scores depend on keys more than on values; measure each separately.


### 9.3 Change what a layer does

- `CFG["layer_types"]` decides which layers are DeltaNet or attention. `Entry` derives `gdn_j` and `att_j` from it,
  so the interface in section 7 follows automatically.
- Chunk boundaries come from `CHUNK` and `plan_chunks` (`scripts/qwen38_ane_model.py`) and `--plan` in the builder.
  Four layers per chunk gives 3 DeltaNet + 1 attention in every chunk.


### 9.4 A different quantization

- The weight path is `QConv`. A new format needs an export in `W[key/...]` (idx, lut, scale, or int8) and a branch in
  `QConv.__init__`. The ANE only accepts vector lookup tables up to 256 values (`docs/QUANTIZATION.md`).
- Rotations must match what the weights were fitted for (`mixer_rotation`, `mlp_rotation`); mismatches mark a build
  `TIMING_ONLY` and the runtime refuses to serve it.


### 9.5 New drafter or sampling rule

- Drafter code: `scripts/dflash2_coreai_drafter.py`, `coreai/dflash2_coreai_build.py`. Taps from the target are chosen
  by `TAPS` and exported by `Entry`.
- Acceptance and sampling: `draft_cycle` in `scripts/qwen38_server.py`; see `docs/research/DFLASH2_SAMPLING_PLAN.md`.


## 10. How to check a change

- Compare against the BF16 reference with the project's KL and top-1 scripts: `scripts/m6_kl512_eval.py`
  (release 0.2 numbers: KL-512 0.052, top-1 agreement 91.9%).
- Long context: `scripts/m6_long_ctx_eval.py` (recall of planted codes; the 0.2.1 notes report all recalled to 195K).
- Per-layer numerical checks: `scripts/m6_attn_core_check.py`, `scripts/m6_gdn_bench.py`, `DBG_O=1` debug outputs.
- Compare the ANE build against a plain-PyTorch version of the same layer with the equations in sections 4.2 and 4.3.
- Watch memory: your dashboard's "Where the 32 GB of memory is" card, and the server log's `[ctx]` lines.


## 11. Gotchas worth knowing

- The ANE compiler fails on tensors wider than 65,536 along one axis (why history is tiled).
- FP16 subnormals and overflow are the usual numeric failures: massive activations in the residual stream,
  tiny layer-0 embeddings, `q * S` products, `softplus`. Check the helpers before adding ops.
- Every macOS update, including minor ones, recompiles all packages (about 55 minutes for a 128K build).
- The first start after a cache purge can take over 30 minutes; the server resumes partial compiles.
- `GDN_FAST=0 ATT_BLOCK=16384` rebuilds the older release graph, useful as a control when something regresses.


## 12. Glossary

- **Token mixer**: the layer part where tokens exchange information (attention or DeltaNet).
- **KV cache**: stored keys and values of earlier tokens, read by every attention layer.
- **Delta rule**: write `beta * (v - what_memory_returns_for_k)` along `k`, so a repeated key replaces its old value.
- **Gated**: the decay `exp(g)` can shrink the whole memory each step; `g` is predicted from the input.
- **GQA**: grouped-query attention, 24 query heads sharing 4 key/value heads.
- **LUT / vector lookup table**: weights stored as small indices into a shared codebook.
- **Prefill / verify**: reading a prompt in large blocks versus checking 8 candidate tokens at a time.
- **Lazy commit**: keeping candidate state as pending until the accepted tokens are known.
